# 🎯 YOLO26 Instance Segmentation

This notebook demonstrates how to use **YOLO26** for instance segmentation tasks using the Ultralytics library.

YOLO26 is the latest generation YOLO model from Ultralytics, featuring:
- ⚡ End-to-end NMS-free inference
- 📦 Optimized edge deployment
- 🎯 State-of-the-art accuracy and speed

> 💡 **Note:** YOLO26 segmentation models use the `-seg` suffix, i.e., `yolo26n-seg.pt`, and are pretrained on COCO (80 classes).
---

## 📋 Table of Contents
1. [Install & Imports](#1-install--imports)
2. [Configuration](#2-configuration)
3. [Setup Data Folder](#3-setup-data-folder)
4. [Load Model](#4-load-model)
5. [Inference on Images](#5-inference-on-images)
6. [Inference on Videos](#6-inference-on-videos)
7. [Export Model](#7-export-model)
8. [Bonus: Train on Custom Dataset](#8-bonus-train-on-custom-dataset)
9. [Bonus: Validate Model](#9-bonus-validate-model)

## 1. Install & Imports

In [ ]:
# Install missing dependencies only when needed
import importlib.util
import subprocess
import sys

required_packages = {
    "ultralytics": "ultralytics",
    "opencv-python": "cv2",
    "matplotlib": "matplotlib",
    "requests": "requests",
    "gdown": "gdown",
    "tqdm": "tqdm",
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
    "onnxslim": "onnxslim",
}

missing = [pkg for pkg, module in required_packages.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("✅ Required packages already installed.")


In [ ]:
# Check available acceleration backends
import shutil
import subprocess
import torch

mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"MPS available   : {mps_available}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
elif mps_available:
    print("Device          : Apple Silicon GPU (MPS)")
else:
    print("Device          : CPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)


In [ ]:
import os
import shutil
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import time
import matplotlib.pyplot as plt
import requests
import torch
from IPython.display import Video, display
from tqdm.notebook import tqdm
from ultralytics import YOLO


## 2. Configuration

> 💡 **Asset source plan:** The demo assets are now hosted directly on `learnopencv.com`, so the notebook downloads them from the published WordPress media URLs. The local filenames stay the same, which keeps the rest of the notebook and the blog post examples aligned.


### ⚙️ Configuration Guide

**🤖 Model Settings:**
- `MODEL_NAME` — The YOLO26 segmentation model to use. Start with `yolo26n-seg.pt` (Nano) for fastest inference and switch to larger variants for better accuracy.
- `IMG_SIZE` — Input image size for the model. Default is `640x640`. Larger sizes may improve accuracy on small objects but will be slower.
- `CONF_THRESHOLD` — Minimum confidence score to consider a detection valid. Range: `0.0 - 1.0`. Lower = more detections, Higher = fewer but more confident.
- `LOW_CONF_THRESHOLD` — Secondary confidence threshold used in the comparison cell.
- `IOU_THRESHOLD` — IoU threshold used during prediction. Range: `0.0 - 1.0`.
- `DEVICE` — Auto-detects `"0"` for CUDA, `"mps"` for Apple Silicon, and `"cpu"` otherwise.

**📥 Download Settings:**
- `SKIP_DOWNLOAD_IF_PRESENT` — Reuse the demo assets if they already exist in `data/input/`, which makes reruns much faster.
- `ASSET_URLS` — Mapping from the local filename used by the notebook to the hosted `learnopencv.com` media URL.
- `DOWNLOAD_HEADERS` — Browser-style request headers used for asset downloads, which helps avoid `403 Forbidden` responses from the hosting layer.
- `EXPECTED_FILES` — Exact asset filenames expected in `data/input/` after download.
- `PREFERRED_VIDEO_ORDER` — Keeps the video examples aligned with the blog post order (`Animals`, `Vehicles`, `Cow`).

**📂 Folder Structure:**
Once you run the setup cell, the following folders will be created automatically:
```
data/
├── input/     ← downloaded images & videos go here
└── output/    ← all segmented results saved here
```


In [ ]:
def detect_device():
    if torch.cuda.is_available():
        return "0"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


@dataclass
class Config:
    # ------------------------------------------------------------------ #
    # Model settings
    # ------------------------------------------------------------------ #
    MODEL_NAME: str            = "yolo26n-seg.pt"   # Options: yolo26n/s/m/l/x-seg.pt
    IMG_SIZE: int              = 640
    CONF_THRESHOLD: float      = 0.25
    LOW_CONF_THRESHOLD: float  = 0.10
    IOU_THRESHOLD: float       = 0.45
    DEVICE: str                = field(default_factory=detect_device)

    # ------------------------------------------------------------------ #
    # Data paths
    # ------------------------------------------------------------------ #
    INPUT_DIR: str             = "data/input"
    OUTPUT_DIR: str            = "data/output"
    SKIP_DOWNLOAD_IF_PRESENT: bool = True

    # ------------------------------------------------------------------ #
    # Hosted demo assets on learnopencv.com
    # ------------------------------------------------------------------ #
    DOWNLOAD_HEADERS: dict[str, str] = field(default_factory=lambda: {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    })
    ASSET_URLS: dict[str, str] = field(default_factory=lambda: {
        "Animals.mp4": "https://learnopencv.com/wp-content/uploads/2026/03/Animals.mp4",
        "Vehicles.mp4": "https://learnopencv.com/wp-content/uploads/2026/03/Vehicles.mp4",
        "Cow.mp4": "https://learnopencv.com/wp-content/uploads/2026/03/Cow.mp4",
        "Common_Objects.png": "https://learnopencv.com/wp-content/uploads/2026/03/Common_Objects-scaled.png",
        "Elephants.png": "https://learnopencv.com/wp-content/uploads/2026/03/Elephants-scaled.png",
        "Traffic.png": "https://learnopencv.com/wp-content/uploads/2026/03/Traffic.png",
        "city.png": "https://learnopencv.com/wp-content/uploads/2026/03/city.png",
    })

    EXPECTED_FILES: list[str] = field(default_factory=lambda: [
        "Animals.mp4",
        "Vehicles.mp4",
        "Cow.mp4",
        "Common_Objects.png",
        "Elephants.png",
        "Traffic.png",
        "city.png",
    ])

    PREFERRED_VIDEO_ORDER: list[str] = field(default_factory=lambda: [
        "Animals.mp4",
        "Vehicles.mp4",
        "Cow.mp4",
    ])


cfg = Config()
print(f"Model           : {cfg.MODEL_NAME}")
print(f"Input size      : {cfg.IMG_SIZE}")
print(f"Confidence      : {cfg.CONF_THRESHOLD}")
print(f"IoU threshold   : {cfg.IOU_THRESHOLD}")
print(f"Device          : {cfg.DEVICE}")
print("Asset source    : learnopencv.com")


## 3. Setup Data Folder 📁

This cell automatically:
- ⬇️ Downloads all images and videos from the hosted `learnopencv.com` asset URLs
- ♻️ Reuses the existing demo assets when they are already present
- ✅ Verifies that every expected blog-post asset is available before inference starts
- 📋 Lists all downloaded files ready for inference


In [ ]:
def setup_folders(cfg):
    """Create data/input and data/output folders."""
    Path(cfg.INPUT_DIR).mkdir(parents=True, exist_ok=True)
    Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    print("✅ Folders ready:")
    print(f"   Input  -> {cfg.INPUT_DIR}")
    print(f"   Output -> {cfg.OUTPUT_DIR}")


def expected_assets_present(cfg):
    existing = {path.name for path in Path(cfg.INPUT_DIR).glob("*") if path.is_file()}
    return all(name in existing for name in cfg.EXPECTED_FILES)


def verify_expected_assets(cfg):
    existing = {path.name for path in Path(cfg.INPUT_DIR).glob("*") if path.is_file()}
    missing = [name for name in cfg.EXPECTED_FILES if name not in existing]
    if missing:
        raise FileNotFoundError(
            "Missing expected demo assets: " + ", ".join(missing)
        )


def ordered_media_files(cfg):
    all_files = [path.name for path in Path(cfg.INPUT_DIR).glob("*") if path.is_file()]
    images = sorted(
        [f for f in all_files if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))],
        key=str.lower,
    )
    video_order = {name: idx for idx, name in enumerate(cfg.PREFERRED_VIDEO_ORDER)}
    videos = sorted(
        [f for f in all_files if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))],
        key=lambda name: (video_order.get(name, len(video_order)), name.lower()),
    )
    return images, videos


def download_assets(cfg):
    """Download the hosted blog assets into data/input/."""
    print("⬇️  Downloading assets from learnopencv.com ...")
    for filename, url in cfg.ASSET_URLS.items():
        save_path = Path(cfg.INPUT_DIR) / filename
        if save_path.exists():
            print(f"   Skipping (already exists): {filename}")
            continue

        print(f"   Downloading: {filename}")
        response = requests.get(
            url,
            headers=cfg.DOWNLOAD_HEADERS,
            stream=True,
            timeout=60,
        )
        try:
            response.raise_for_status()
            with save_path.open("wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
        except requests.HTTPError as exc:
            raise RuntimeError(
                f"Failed to download '{filename}' from '{url}'."
            ) from exc
        finally:
            response.close()
        print(f"   Saved: {save_path}")
    print("✅ Asset download complete!")


def setup_data(cfg):
    """Create folders, download files if needed, and list all inputs."""
    setup_folders(cfg)
    print("📦 Asset source: learnopencv.com")

    if cfg.SKIP_DOWNLOAD_IF_PRESENT and expected_assets_present(cfg):
        print("✅ Demo assets already present — skipping download.")
    else:
        download_assets(cfg)

    verify_expected_assets(cfg)
    images, videos = ordered_media_files(cfg)

    print(f"\n📂 Files ready in {cfg.INPUT_DIR}:")
    print(f"   🖼️  Images ({len(images)}) : {images}")
    print(f"   🎥 Videos ({len(videos)}) : {videos}")

    return images, videos


images, videos = setup_data(cfg)


## 4. Load Model 🤖

YOLO26 offers 5 model sizes — choose based on your speed vs accuracy needs:

| Model | Size | mAP box 50-95 | mAP mask 50-95 | Speed T4 TensorRT (ms) | Params (M) |
|---|---|---|---|---|---|
| yolo26n-seg | Nano   | 39.6 | 33.9 | 2.1  | 2.7  |
| yolo26s-seg | Small  | 47.3 | 40.0 | 3.3  | 10.4 |
| yolo26m-seg | Medium | 52.5 | 44.1 | 6.7  | 23.6 |
| yolo26l-seg | Large  | 54.4 | 45.5 | 8.0  | 28.0 |
| yolo26x-seg | XLarge | 56.5 | 47.0 | 16.4 | 62.8 |

> 💡 **Tip:** Start with `yolo26n-seg.pt` (Nano) for fastest inference. Switch to larger models for higher accuracy. Change `MODEL_NAME` in the Config above.

> 📝 **Benchmark note:** The speed column above is the official **T4 TensorRT** benchmark at `640`, and the parameter counts are the fused-model counts reported in the Ultralytics table.


In [ ]:
# Load pretrained YOLO26 segmentation model
model = YOLO(cfg.MODEL_NAME)
model.info()

## 5. Inference on Images 🖼️

Each image from `data/input/` is processed individually.
Results (original vs segmented) are displayed side by side and saved to `data/output/`.

> 💡 **Tip:** Adjust `CONF_THRESHOLD` in Config to control detection sensitivity. Higher value = fewer but more confident detections.

In [ ]:
def run_inference_single_image(model, cfg, img_file):
    """
    Run YOLO26 segmentation on a single image.
    Displays original vs segmented side by side.
    Saves annotated result to data/output/

    Args:
        model    : Loaded YOLO26 model
        cfg      : Config object
        img_file : Image filename (from data/input/)
    """
    img_path = os.path.join(cfg.INPUT_DIR, img_file)
    print(f"🔍 Processing: {img_file}")

    results = model(
        source=img_path,
        conf=cfg.CONF_THRESHOLD,
        iou=cfg.IOU_THRESHOLD,
        imgsz=cfg.IMG_SIZE,
        device=cfg.DEVICE,
        verbose=False
    )

    result = results[0]

    # Original image
    original  = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)

    # Annotated image with masks, boxes and labels
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

    # Save to data/output/
    save_path = os.path.join(cfg.OUTPUT_DIR, f"seg_{img_file}")
    cv2.imwrite(save_path, result.plot())

    # Plot side by side
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(original)
    axes[0].set_title(f"Original: {img_file}", fontsize=13)
    axes[0].axis("off")

    axes[1].imshow(annotated)
    axes[1].set_title(f"YOLO26 Segmentation | Objects detected: {len(result.boxes)}", fontsize=13)
    axes[1].axis("off")

    plt.suptitle("YOLO26 Instance Segmentation", fontsize=15)
    plt.tight_layout()
    plt.show()

    print(f"   ✅ Objects detected : {len(result.boxes)}")
    print(f"   💾 Saved to         : {save_path}")
    print()


# Run inference on each image one by one
if images:
    for img_file in images:
        run_inference_single_image(model, cfg, img_file)
else:
    print("⚠️  No images found in data/input/ — please check your data setup.")

### 5.1 🔍 Effect of Confidence Threshold

Let's see what happens when we **lower the confidence threshold**!

In [ ]:
# Try with a lower confidence threshold and compare results
LOW_CONF = cfg.LOW_CONF_THRESHOLD

if images:
    for img_file in images:
        img_path = os.path.join(cfg.INPUT_DIR, img_file)

        # Original confidence
        results_default = model(
            source=img_path,
            conf=cfg.CONF_THRESHOLD,
            iou=cfg.IOU_THRESHOLD,
            imgsz=cfg.IMG_SIZE,
            device=cfg.DEVICE,
            verbose=False
        )

        # Lower confidence
        results_low = model(
            source=img_path,
            conf=LOW_CONF,
            iou=cfg.IOU_THRESHOLD,
            imgsz=cfg.IMG_SIZE,
            device=cfg.DEVICE,
            verbose=False
        )

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        axes[0].imshow(cv2.cvtColor(results_default[0].plot(), cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Confidence = {cfg.CONF_THRESHOLD} | Objects: {len(results_default[0].boxes)}", fontsize=12)
        axes[0].axis("off")

        axes[1].imshow(cv2.cvtColor(results_low[0].plot(), cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Confidence = {LOW_CONF} | Objects: {len(results_low[0].boxes)}", fontsize=12)
        axes[1].axis("off")

        plt.suptitle(f"Confidence Threshold Comparison — {img_file}", fontsize=14)
        plt.tight_layout()
        plt.show()
else:
    print("⚠️  No images available for the confidence-threshold comparison.")


## 6. Inference on Videos 🎥

Each video is selected by filename so the notebook stays aligned with the blog post examples (`Animals`, `Vehicles`, then `Cow`). Processed videos are saved to `data/output/`.


In [ ]:
def run_inference_single_video(model, cfg, vid_file):
    vid_path = os.path.join(cfg.INPUT_DIR, vid_file)
    output_path = os.path.join(cfg.OUTPUT_DIR, f"seg_{vid_file}")
    temp_path = os.path.join(cfg.OUTPUT_DIR, f"temp_{vid_file}")
    use_ffmpeg = shutil.which("ffmpeg") is not None

    print(f"🎥 Processing video: {vid_file}")

    cap = cv2.VideoCapture(vid_path)
    if not cap.isOpened():
        raise RuntimeError(f"Unable to open video: {vid_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    fps = fps if fps > 0 else 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"   📐 Resolution : {width}x{height} @ {fps:.2f}fps | {total} frames")

    writer_path = temp_path if use_ffmpeg else output_path
    writer = cv2.VideoWriter(
        writer_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Unable to create output video: {writer_path}")

    progress_total = total if total > 0 else None

    # --- METRICS TRACKING INITIALIZATION ---
    start_time = time.time()
    total_inference_time = 0.0
    frames_processed = 0

    # <-- UPDATED: Removed ncols=80 so it perfectly fits the Jupyter output cell
    with tqdm(total=progress_total, desc="   ⏳ Processing", unit="frame", colour="green") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Measure exactly how long the model takes (Latency)
            t0 = time.time()
            results = model(
                frame,
                conf=cfg.CONF_THRESHOLD,
                iou=cfg.IOU_THRESHOLD,
                imgsz=cfg.IMG_SIZE,
                device=cfg.DEVICE,
                verbose=False,
            )
            t1 = time.time()
            total_inference_time += (t1 - t0)

            writer.write(results[0].plot())
            pbar.update(1)
            frames_processed += 1

    cap.release()
    writer.release()

    # --- CALCULATE AND PRINT METRICS ---
    total_duration = time.time() - start_time
    avg_latency_ms = (total_inference_time / frames_processed) * 1000 if frames_processed > 0 else 0
    throughput_fps = frames_processed / total_duration if total_duration > 0 else 0

    print(f"\n   ⏱️  Total Pipeline Duration : {total_duration:.2f} seconds")
    print(f"   🚀 End-to-End Throughput  : {throughput_fps:.2f} FPS")
    print(f"   ⚡ Average Model Latency  : {avg_latency_ms:.2f} ms/frame")

    # --- FFmpeg processing ---
    if use_ffmpeg:
        print("   🎞️ Re-encoding with ffmpeg...")
        result = subprocess.run(
            [
                "ffmpeg", "-y",
                "-i", temp_path,
                "-vcodec", "libx264",
                "-crf", "23",
                output_path,
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        if result.returncode == 0:
            os.remove(temp_path)
        else:
            print("   ⚠️ ffmpeg re-encode failed, keeping raw MP4V output instead.")
            os.replace(temp_path, output_path)
    else:
        print("   ⚠️ ffmpeg not found, keeping raw MP4V output.")

    print(f"   ✅ Saved to: {output_path}\n")
    return output_path

- 🎥 **Video 1 — Animals.mp4** — A scene with deer moving around in their natural habitat.

In [ ]:
# ▶️ Video 1 — Animals.mp4
video_name = "Animals.mp4"
if video_name in videos:
    output_path = run_inference_single_video(model, cfg, video_name)
    display(Video(output_path, embed=True, width=800))
else:
    print(f"⚠️  {video_name} not found in data/input/")


> 📝 *Note: You may notice the deer are being detected as **"sheep"** — this is expected behavior! YOLO26 is pretrained on the **COCO dataset** which does not include a "deer" class. As a result, the model tries to match deer to the closest class it knows, which in this case is "sheep". This is a great example of why **fine-tuning on a custom dataset** is important when working with classes not present in COCO.*

- 🎥 **Video 2 — Vehicles.mp4** — A highway scene with fast-moving cars passing by. This is a great test for YOLO26 segmentation as it challenges the model with **fast-moving objects** and **varying object sizes** at different distances.

In [ ]:
# ▶️ Video 2 — Vehicles.mp4
video_name = "Vehicles.mp4"
if video_name in videos:
    output_path = run_inference_single_video(model, cfg, video_name)
    display(Video(output_path, embed=True, width=800))
else:
    print(f"⚠️  {video_name} not found in data/input/")


- 🎥 **Video 3 — Cow.mp4** — A scene with cows passing by and overlapping each other. This is an excellent test for **occlusion handling** — where objects partially cover one another, challenging the model to maintain accurate segmentation masks for each individual animal.

In [ ]:
# ▶️ Video 3 — Cow.mp4
video_name = "Cow.mp4"
if video_name in videos:
    output_path = run_inference_single_video(model, cfg, video_name)
    display(Video(output_path, embed=True, width=800))
else:
    print(f"⚠️  {video_name} not found in data/input/")


## 7. Export Model 📦

Export your YOLO26 model to any supported format for deployment using the `format` argument.

> 💡 **ProTip:** Export to [ONNX](https://docs.ultralytics.com/integrations/onnx/) or [OpenVINO](https://docs.ultralytics.com/integrations/openvino/) for up to **3x CPU speedup.**

> 💡 **ProTip:** Export to [TensorRT](https://docs.ultralytics.com/integrations/tensorrt/) for up to **5x GPU speedup.**

| Format | `format` Argument | Model | Metadata |
|---|---|---|---|
| PyTorch | - | `yolo26n-seg.pt` | ✅ |
| TorchScript | `torchscript` | `yolo26n-seg.torchscript` | ✅ |
| ONNX | `onnx` | `yolo26n-seg.onnx` | ✅ |
| OpenVINO | `openvino` | `yolo26n-seg_openvino_model/` | ✅ |
| TensorRT | `engine` | `yolo26n-seg.engine` | ✅ |
| CoreML | `coreml` | `yolo26n-seg.mlpackage` | ✅ |
| TF SavedModel | `saved_model` | `yolo26n-seg_saved_model/` | ✅ |
| TF GraphDef | `pb` | `yolo26n-seg.pb` | ❌ |
| TF Lite | `tflite` | `yolo26n-seg.tflite` | ✅ |
| TF Edge TPU | `edgetpu` | `yolo26n-seg_edgetpu.tflite` | ✅ |
| TF.js | `tfjs` | `yolo26n-seg_web_model/` | ✅ |
| PaddlePaddle | `paddle` | `yolo26n-seg_paddle_model/` | ✅ |
| MNN | `mnn` | `yolo26n-seg.mnn` | ✅ |
| NCNN | `ncnn` | `yolo26n-seg_ncnn_model/` | ✅ |
| RKNN | `rknn` | `yolo26n-seg_rknn_model/` | ✅ |

In [ ]:
# --- 1. Export the model to ONNX format ---
# Assuming 'model' and 'cfg' are already defined in previous cells
export_path = model.export(format="onnx", imgsz=cfg.IMG_SIZE)
print(f"✅ Model exported to: {export_path}")


# --- 2. Test the exported ONNX model ---
# Use the 'images' list from your previous data loading setup
if images:
    # Load the specific ONNX model file
    onnx_model = YOLO(export_path)
    test_img_path = os.path.join(cfg.INPUT_DIR, images[1])

    # Run inference using ONNX Runtime
    onnx_results = onnx_model(test_img_path, conf=cfg.CONF_THRESHOLD)

    # Get the single result object
    result = onnx_results[0]
    print(f"✅ ONNX inference successful! Detected {len(result.boxes)} objects.")

    original_img_bgr = result.orig_img
    original_img_rgb = cv2.cvtColor(original_img_bgr, cv2.COLOR_BGR2RGB)

    annotated_img_bgr = result.plot()
    annotated_img_rgb = cv2.cvtColor(annotated_img_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(16, 9)) # Setup 1 row, 2 columns

    axes[0].imshow(original_img_rgb)
    axes[0].set_title(f"Original: {images[0]}", fontsize=14)
    axes[0].axis('off') # Hide axis ticks

    axes[1].imshow(annotated_img_rgb)
    axes[1].set_title(f"YOLO26 ONNX Segmentation | Objects: {len(result.boxes)}", fontsize=14)
    axes[1].axis('off') # Hide axis ticks

    plt.tight_layout()
    plt.show() # Display the plot in the notebook

else:
    print("⚠️  No images available in data/input/ to test the exported ONNX model.")

## 8. 🎁 Bonus: Train on Custom Dataset

Want to train YOLO26 on your own dataset? Here's how!

### Step 1 — Prepare your dataset in YOLO segmentation format:
```
dataset/
├── train/
│   ├── images/
│   └── labels/
└── valid/
    ├── images/
    └── labels/
```

### Step 2 — Create a dataset YAML file:
```yaml
path: /path/to/dataset
train: train/images
val:   valid/images
names:
  0: class1
  1: class2
```

### Step 3 — Run training:
Uncomment and run the cell below. Training outputs (weights, plots, metrics) will be saved to `data/output/train/`.

> 💡 **Tip:** Use `yolo26n-seg.pt` for faster training. Use `yolo26l-seg.pt` or `yolo26x-seg.pt` for better accuracy on complex datasets.

> 💡 **Tip:** Use tools like [Roboflow](https://roboflow.com) or [LabelMe](https://github.com/labelmeai/labelme) to annotate your dataset in YOLO segmentation format easily.

In [ ]:
# ⚠️ Uncomment the lines below to train on your custom dataset

# EPOCHS    = 50    # Number of training epochs
# BATCH     = 16    # Batch size (reduce if you run out of GPU memory)
# DATA_YAML = "path/to/your_dataset.yaml"

# train_model = YOLO(cfg.MODEL_NAME)
# results = train_model.train(
#     data    = DATA_YAML,
#     epochs  = EPOCHS,
#     imgsz   = cfg.IMG_SIZE,
#     batch   = BATCH,
#     device  = cfg.DEVICE,
#     project = cfg.OUTPUT_DIR,
#     name    = "train",
#     save    = True,
#     plots   = True,
# )
# print(f"Training complete! Results saved to: {cfg.OUTPUT_DIR}/train")

## 9. 🎁 Bonus: Validate Model

After training, validate your model's performance on a dataset.

Key metrics to look at:
- **mAP50 (boxes)** — detection accuracy at IoU threshold 0.5
- **mAP50-95 (boxes)** — detection accuracy averaged across IoU thresholds 0.5 to 0.95
- **mAP50 (masks)** — segmentation mask accuracy at IoU threshold 0.5
- **mAP50-95 (masks)** — segmentation mask accuracy averaged across IoU thresholds

> 💡 **Tip:** Higher mAP50-95 is a better indicator of real-world performance than mAP50 alone.

> 💡 **Tip:** To validate your custom trained model, replace `cfg.MODEL_NAME` with the path to your best checkpoint, e.g. `data/output/train/weights/best.pt`.

In [ ]:
# ⚠️ Uncomment the lines below to validate your model

# metrics = model.val(
#     data  = "coco8-seg.yaml",   # Replace with your dataset YAML
#     imgsz = cfg.IMG_SIZE,
#     device= cfg.DEVICE
# )

# print("=== Validation Metrics ===")
# print(f"mAP50    (boxes) : {metrics.box.map50:.4f}")
# print(f"mAP50-95 (boxes) : {metrics.box.map:.4f}")
# print(f"mAP50    (masks) : {metrics.seg.map50:.4f}")
# print(f"mAP50-95 (masks) : {metrics.seg.map:.4f}")